# Yoghurt Sensory Clustering — Unsupervised Product Segmentation

**Author:** Sanjeev Malhotra  
**Context:** Master of Food Science & Technology, University of Queensland

Unsupervised clustering of descriptive-panel yoghurt data (24 samples, 7 sensory attributes).
**k-means** (k=3, silhouette 0.59) and **Ward hierarchical clustering** both independently identify
three product segments — *indulgent/creamy*, *tart & fruity*, and *plain/lean* — validated on a
PCA map capturing ~85% of variance.

**Tools:** pandas · scikit-learn · scipy · matplotlib

---
### Method
1. Standardise the 7 sensory attributes (z-score) so no attribute dominates on scale alone.
2. Choose the number of clusters *k* using the elbow (inertia) and silhouette score.
3. Fit k-means, then project to 2-D with PCA to visually validate separation.
4. Confirm the structure independently with hierarchical clustering (dendrogram).
5. Interpret each cluster's mean profile into a named product segment.

## 1. Load and standardise the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Load the descriptive-panel data
df = pd.read_csv("yoghurt_sensory.csv")

# Separate sample labels from the numeric attribute matrix
sample_names = df["Sample"]
X = df.drop(columns=["Sample"])

# Standardise: mean 0, sd 1 per attribute (PCA and k-means are both variance/distance driven)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Data shape:", X.shape)
df.head()

## 2. Choose the number of clusters (k)

k-means requires *k* as an input. Two complementary diagnostics:
- **Elbow** — total within-cluster spread (`inertia`) vs k; look for the bend where extra clusters stop helping.
- **Silhouette** — how tight-and-separated the clusters are (−1 to 1); pick the k that maximises it.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias, sils = [], []
K = range(2, 9)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(K, inertias, "o-"); ax[0].set_title("Elbow"); ax[0].set_xlabel("k"); ax[0].set_ylabel("inertia")
ax[1].plot(K, sils, "o-");     ax[1].set_title("Silhouette"); ax[1].set_xlabel("k"); ax[1].set_ylabel("score")
plt.tight_layout(); plt.show()

for k, s in zip(K, sils):
    print(f"k={k}: silhouette={s:.3f}")

**Result:** silhouette peaks at **k = 3** (~0.59) and the elbow bends at 3 — both point to three clusters.
A silhouette above ~0.5 indicates genuinely well-separated groups.

## 3. Fit k-means (k=3) and validate on a PCA map

In [ ]:
from sklearn.decomposition import PCA

# Final clustering
km = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = km.fit_predict(X_scaled)

# PCA to 2-D for visual validation
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=clusters, cmap="viridis", s=80)
for i, name in enumerate(sample_names):
    plt.annotate(name, (coords[i, 0], coords[i, 1]), fontsize=8)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)")
plt.title("Yoghurt sensory clusters (k=3)")
plt.tight_layout(); plt.show()

print(f"Variance captured by PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.0f}%")

The three clusters occupy clearly separate regions of the PCA map, which captures ~85% of the total
variance — so the visual separation is trustworthy, not an artefact of a low-information projection.

## 4. Interpret the clusters

A cluster label (0/1/2) is meaningless until described. Averaging the **original (unscaled)** attribute
scores within each cluster gives each group's sensory personality.

In [ ]:
profile = X.copy()
profile["Cluster"] = clusters
cluster_means = profile.groupby("Cluster").mean().round(1)
cluster_means

**Named segments (read from the cluster-means table):**

| Cluster | Profile | Signature attributes |
|---|---|---|
| Indulgent / creamy | full-fat, dessert-style | high sweetness, creaminess, thickness; low sourness |
| Tart & fruity | sharper, fruit-flavoured | high sourness & fruity aroma; some astringency; lower creaminess |
| Plain / lean | natural / low-fat | low sweetness, creaminess and fruit; moderate sourness |

This is a **product map** a developer could use to spot gaps in a yoghurt range.

## 5. Independent confirmation — hierarchical clustering

Hierarchical clustering starts with every sample as its own group and repeatedly merges the two closest
groups, recording the history as a **dendrogram**. Reading a horizontal cut across the tallest vertical
gap reproduces the same group count — a cross-check that does not assume *k* in advance.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(X_scaled, method="ward")   # Ward merges to keep clusters compact

plt.figure(figsize=(11, 5))
dendrogram(Z, labels=sample_names.values)
plt.title("Hierarchical clustering (Ward)")
plt.xlabel("Sample"); plt.ylabel("Merge distance")
plt.tight_layout(); plt.show()

Cutting across the tall gap near the top yields **three branches**, matching k-means. Two independent
algorithms agreeing on the same three segments makes the result credible.

---
## Conclusion
Descriptive-panel data separates these yoghurts into three well-defined sensory segments
(indulgent/creamy, tart & fruity, plain/lean), consistent across k-means and hierarchical clustering and
visible on a high-variance PCA map. The same workflow applies to any sensory or consumer dataset for
product segmentation and gap analysis.